# Self-Supervised Learning for Named Entity Recognition (NER)

## 📌 Objective

This project implements a **Self-Supervised Learning framework** for **Named Entity Recognition (NER)** using a Transformer-based architecture (DistilBERT).

The training pipeline consists of two stages:

1. **Self-Supervised Pretraining (Masked Language Modeling - MLM)**  
2. **Supervised Fine-Tuning for NER**

The goal is to leverage large-scale unlabeled text to learn strong contextual representations before adapting the model to a downstream token classification task.

---

## 🧠 Project Overview

Modern NLP systems use self-supervised learning to pretrain language models on massive unlabeled corpora. These pretrained models are then fine-tuned on specific tasks like NER.

In this project:

- WikiText-2 is used for **MLM pretraining**
- CoNLL-2003 is used for **NER fine-tuning**
- DistilBERT serves as the backbone Transformer model

---

## 🚀 Stage 1: Self-Supervised Pretraining (MLM)

### Dataset
- **WikiText-2 (raw version)**
- Unlabeled text corpus

### Pretext Task
- **Masked Language Modeling (MLM)**
- Randomly masks 15% of tokens
- Model predicts masked tokens using context

### Model
- `DistilBertForMaskedLM`
- Pretrained `distilbert-base-uncased`
- Fine-tuned for 1 epoch

### Outcome
The model learns:
- Contextual word representations
- Semantic relationships
- Long-range dependencies

---

## 🏷 Stage 2: Supervised NER Fine-Tuning

### Dataset
- **CoNLL-2003**
- Standard benchmark for NER

### Entity Classes
- **PER** – Person
- **ORG** – Organization
- **LOC** – Location
- **MISC** – Miscellaneous

### Model
- `DistilBertForTokenClassification`
- Loads encoder weights from MLM phase
- Adds classification head

### Training
- 2 epochs
- F1 score used for evaluation

---

## 📊 Results

| Metric | Value |
|--------|--------|
| Validation F1 | **0.93** |
| Validation Loss | 0.058 |
| Precision (avg) | 0.93 |
| Recall (avg) | 0.94 |

### Per-Entity Performance

- PER → 0.97 F1
- LOC → 0.95 F1
- ORG → 0.90 F1
- MISC → 0.83 F1

The model demonstrates strong contextual understanding and accurate entity classification.

---

## 🔍 Example Inference

Input:

```

A person was born in Hyderabad and worked at Microsoft.

```

Predicted Entities:

- Hyderabad → LOC  
- Microsoft → ORG  

---

## 🏗 Architecture Summary

```

Unlabeled Text (WikiText-2)
↓
Masked Language Modeling (MLM)
↓
Pretrained DistilBERT Encoder
↓
Add Token Classification Head
↓
Fine-Tune on CoNLL-2003
↓
Named Entity Recognition

```

---

## 📁 Deliverables

- ✅ Data preprocessing pipeline
- ✅ Self-supervised MLM pretraining
- ✅ Transformer-based architecture
- ✅ NER fine-tuning
- ✅ F1 evaluation & classification report
- ✅ Inference pipeline
- ✅ Saved trained model

---

## 🎯 Conclusion

This project successfully demonstrates:

- Self-supervised representation learning
- Transfer learning using Transformers
- Token-level classification for NER
- End-to-end training and deployment

The achieved F1 score (~93%) confirms the effectiveness of combining self-supervised pretraining with supervised fine-tuning.

---

### 🔥 Technologies Used

- PyTorch
- HuggingFace Transformers
- HuggingFace Datasets
- SeqEval
- Google Colab (GPU)

## **Code:**

In [ ]:
# ============================================================
# SELF-SUPERVISED LEARNING FOR NER
# (MLM PRE-TRAINING + NER FINE-TUNING)
# ============================================================

!pip install -U transformers datasets accelerate seqeval

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForMaskedLM,
    DistilBertForTokenClassification,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    pipeline
)
from seqeval.metrics import f1_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------
# 1. Load Unlabeled Dataset (Self-Supervised Phase - MLM)
# ------------------------------------------------------------

print("\nLoading WikiText-2 Dataset for MLM...\n")
mlm_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print(mlm_dataset)

# ------------------------------------------------------------
# 2. Tokenization for MLM
# ------------------------------------------------------------

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_mlm(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_mlm = mlm_dataset.map(
    tokenize_mlm,
    batched=True,
    remove_columns=["text"]
)

tokenized_mlm.set_format("torch")
print("\nMLM Dataset Tokenized Successfully.\n")

# ------------------------------------------------------------
# 3. Define Pretext Task (Masked Language Modeling)
# ------------------------------------------------------------

mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# ------------------------------------------------------------
# 4. Initialize Transformer Model (DistilBERT)
# ------------------------------------------------------------

mlm_model = DistilBertForMaskedLM.from_pretrained(
    "distilbert-base-uncased"
)
mlm_model.to(device)

# ------------------------------------------------------------
# 5. Self-Supervised Pre-Training (MLM)
# ------------------------------------------------------------

mlm_training_args = TrainingArguments(
    output_dir="./distilbert-mlm-pretrained",
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=100,
    eval_strategy="no",
    fp16=torch.cuda.is_available(),
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_training_args,
    train_dataset=tokenized_mlm["train"],
    eval_dataset=tokenized_mlm["validation"],
    data_collator=mlm_collator,
)

print("\nStarting Self-Supervised Pre-training...\n")
mlm_trainer.train()

print("\nSaving Pre-trained Model...\n")
mlm_model.save_pretrained("./distilbert-mlm-pretrained")
tokenizer.save_pretrained("./distilbert-mlm-pretrained")

# ============================================================
# SUPERVISED PHASE: NER FINE-TUNING
# ============================================================

# ------------------------------------------------------------
# 6. Load Labeled Dataset (CoNLL-2003)  [2026 SAFE VERSION]
# ------------------------------------------------------------

print("\nDownloading Official CoNLL-2003 Dataset...\n")

!wget -q https://data.deepai.org/conll2003.zip

import zipfile
with zipfile.ZipFile("conll2003.zip", "r") as zip_ref:
    zip_ref.extractall("conll2003_data")

from datasets import Dataset, DatasetDict

def read_conll_file(filepath):
    tokens = []
    ner_tags = []
    current_tokens = []
    current_tags = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_tokens:
                    tokens.append(current_tokens)
                    ner_tags.append(current_tags)
                    current_tokens = []
                    current_tags = []
            else:
                splits = line.split()
                if len(splits) >= 4:
                    current_tokens.append(splits[0])
                    current_tags.append(splits[-1])

    return {"tokens": tokens, "ner_tags": ner_tags}

train_data = read_conll_file("conll2003_data/train.txt")
valid_data = read_conll_file("conll2003_data/valid.txt")
test_data  = read_conll_file("conll2003_data/test.txt")

ner_dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(valid_data),
    "test": Dataset.from_dict(test_data),
})

print(ner_dataset)

# Build label list
label_list = sorted(list(set(tag for tags in train_data["ner_tags"] for tag in tags)))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label_list)

# Encode labels
def encode_labels(example):
    example["ner_tags"] = [label2id[tag] for tag in example["ner_tags"]]
    return example

ner_dataset = ner_dataset.map(encode_labels)

# ------------------------------------------------------------
# 7. Tokenization & Label Alignment for NER
# ------------------------------------------------------------

def tokenize_ner(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx])
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_ner = ner_dataset.map(tokenize_ner, batched=True)
tokenized_ner.set_format("torch")

print("\nNER Dataset Tokenized Successfully.\n")

# ------------------------------------------------------------
# 8. Load Pre-trained Encoder for NER
# ------------------------------------------------------------

ner_model = DistilBertForTokenClassification.from_pretrained(
    "./distilbert-mlm-pretrained",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

ner_model.to(device)

# ------------------------------------------------------------
# 9. Fine-Tune for NER (Supervised Training)
# ------------------------------------------------------------

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [id2label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return {"f1": f1_score(true_labels, true_predictions)}

ner_training_args = TrainingArguments(
    output_dir="./distilbert-ner-model",
    learning_rate=3e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    logging_steps=100,
    eval_strategy="epoch",
    fp16=torch.cuda.is_available(),
)

ner_trainer = Trainer(
    model=ner_model,
    args=ner_training_args,
    train_dataset=tokenized_ner["train"],
    eval_dataset=tokenized_ner["validation"],
    compute_metrics=compute_metrics,
)

print("\nStarting NER Fine-Tuning...\n")
ner_trainer.train()

print("\nEvaluating NER Model...\n")
eval_results = ner_trainer.evaluate()
print("\nNER Evaluation Results:\n", eval_results)

# ------------------------------------------------------------
# 10. Detailed Classification Report
# ------------------------------------------------------------

predictions, labels, _ = ner_trainer.predict(tokenized_ner["validation"])
preds = np.argmax(predictions, axis=2)

true_labels = [
    [label_list[l] for l in label if l != -100]
    for label in labels
]
true_predictions = [
    [label_list[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]

print("\nClassification Report:\n")
print(classification_report(true_labels, true_predictions))

# ------------------------------------------------------------
# 11. Inference Pipeline (Deployment Ready)
# ------------------------------------------------------------

print("\nRunning NER Inference...\n")

ner_pipeline = pipeline(
    "token-classification",
    model=ner_model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

test_sentence = "A person was born in Hyderabad and worked at Microsoft."
results = ner_pipeline(test_sentence)

print("Input:", test_sentence)
print("\nPredicted Entities:")
for r in results:
    print(f"Entity: {r['word']}, Label: {r['entity_group']}, Confidence: {round(r['score'],4)}")

# ------------------------------------------------------------
# 12. Save Final Model
# ------------------------------------------------------------

ner_model.save_pretrained("./final_ner_model")
tokenizer.save_pretrained("./final_ner_model")

# ------------------------------------------------------------
# Deliverables Summary
# ------------------------------------------------------------

print("\n" + "="*60)
print("SELF-SUPERVISED NER PROJECT - DELIVERABLES REPORT")
print("="*55)
print("(a) Data & Preprocessing Pipeline: COMPLETED")
print("(b) Self-Supervised Model (MLM): PRE-TRAINED")
print("(c) NER Fine-Tuned Model: TRAINED & SAVED")
print("(d) Evaluation: F1 Score + Classification Report Generated")
print("(e) Deployment: Inference Pipeline Ready")
print("="*55)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=47b9ba1daf3e4c06870d7b44d95a80c7d60840288c4e1b9bbeb5082497276fb0
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformer

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating NER Model...




NER Evaluation Results:
 {'eval_loss': 0.058185599744319916, 'eval_f1': 0.931241655540721, 'eval_runtime': 3.6725, 'eval_samples_per_second': 943.778, 'eval_steps_per_second': 59.088, 'epoch': 2.0}

Classification Report:

              precision    recall  f1-score   support

         LOC       0.95      0.96      0.95      2618
        MISC       0.83      0.83      0.83      1231
         ORG       0.89      0.91      0.90      2056
         PER       0.97      0.98      0.97      3029

   micro avg       0.93      0.94      0.93      8934
   macro avg       0.91      0.92      0.91      8934
weighted avg       0.93      0.94      0.93      8934


Running NER Inference...

Input: A person was born in Hyderabad and worked at Microsoft.

Predicted Entities:
Entity: hyderabad, Label: LOC, Confidence: 0.9926999807357788
Entity: microsoft, Label: ORG, Confidence: 0.9693999886512756


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


SELF-SUPERVISED NER PROJECT - DELIVERABLES REPORT
(a) Data & Preprocessing Pipeline: COMPLETED
(b) Self-Supervised Model (MLM): PRE-TRAINED
(c) NER Fine-Tuned Model: TRAINED & SAVED
(d) Evaluation: F1 Score + Classification Report Generated
(e) Deployment: Inference Pipeline Ready
